## 필독!!!

<h3> 여기 있는 코드는 절대 실행하지 마십시오. </h3>

눈으로만 보고 이해하시거나

복붙하셔서 실제 linux나 파이썬 환경에서 실행해 주시기 바랍니다.

이 파일은 jupyter 파일입니다.

여기 있는 코드는 모두 jupyter가 아닌 실제 파이썬 및 ROS2 환경에서 사용할 수 있는 코드로 작성하였습니다.

ROS2 코드를 jupyter에서 실행하는 방법이 없는 것은 아니나 별도의 방법이 따로 존재하기 때문에(`jupyter_ws` 참고)

여기 있는 코드를 실행하게 될 경우 일부 오류나 무한루프 등에 빠질 수 있는 위험이 있습니다.

### 설치 및 실행

```bash
sudo apt update
sudo apt install ros-humble-gazebo-ros-pkgs
```

실행

```bash
gazebo
```

![Gazebo](../imgs/gazebo/gazebo_nothing.png)

기본조작

1. 마우스 좌클릭을 누른 채로 이동하면 화면을 이동할 수 있다.
2. 마우스 스크롤를 누른 채로 이동하면 공간을 회전시킬 수 있다.
3. 마우스 스크롤을 위아래로 굴리거나, 마우스 우클릭을 한 채로 이동하면 화면 확대/축소 가능하다.

![Gazebo_basic](../imgs/gazebo/gazebo_basic1.png)

초록색 박스 구역의 기본 모델을 배치할 수도 있으며

빨간색 박스의 `insert`탭에 가면 기본 모델들을 사용해 배치할 수도 있다.

모델 삭제는 삭제할 모델을 누른 후 delete 키를 누르면 삭제할 수 있다.

세계를 완성하고 나면

File > Save World As를 누르거나 또는 Ctrl + Shift + s를 눌러

새 파일 이름을 입력하여 .sdf 형식으로 저장 가능하다.

여기서 sdf란

urdf는 로봇 자체 위주 정의 파일이지만, sdf는 world 전체, 센서, 플러그인, 조명 등등

더 많은 요소를 포함하여 저장할 수 있는 형식으로

urdf는 gazebo에서 사용할 때 변환이 필요하지만

sdf는 직접 사용이 가능하다.

my_world의 이름으로(확장자는 sdf 선택) 저장하고

gazebo를 종료시킨 뒤

```bash
gazebo my_world
```

를 실행시켜 저장된 모델이 잘 나오는지 확인한다.

(보통 gazebo를 실행시킨 위치에 저장되므로 경로를 잘 확인해야 한다.)

### ROS2 와 Gazebo 연동 테스트

터미널을 2개를 열고

한쪽에는

```bash
gazebo --verbose /opt/ros/humble/share/gazebo_plugins/worlds/gazebo_ros_diff_drive_demo.world
```

를 실행시켜 로봇이 잘 나오는지 확인하고

반대쪽 터미널에

```bash
ros2 topic pub /demo/cmd_demo geometry_msgs/msg/Twist "{linear: {x: 0.1, y: 0, z: 0}, angular: {x: 0, y: 0, z: 0.1}}" -r 10
```

를 실행시켜 Gazebo에서 로봇이 잘 이동하는지 확인한다.

이번엔 또하나의 터미널을 열어

`rviz2`를 실행한다.

![rviz](../imgs/gazebo/gazebo_rviz2_1.png)

사진과 같이 왼쪽 하단에 add 버튼을 눌러

rviz_default_plugins 아래의 TF를 선택하고 OK 버튼을 눌러 추가해 준 다음

![rviz](../imgs/gazebo/gazebo_rviz2_2.png)

fixed_frame을 `odom_demo` 토픽으로 바꿔준다.

#### 연동 테스트 2

이전 작업을 모두 종료하고

첫 번째  터미널에

```bash
gazebo --verbose /opt/ros/humble/share/gazebo_plugins/worlds/gazebo_ros_ackermann_drive_demo.world
```

를 실행한다.

첫 실행이면 모델 로드하는데 상당한 시간이 걸릴 수 있으므로 기다려준다.

이후 또 다른 터미널에서

```bash
ros2 topic pub /demo/cmd_demo geometry_msgs/msg/Twist "{linear: {x: 1.0}, angular: {z: 0.3}}"
```

를 실행하여 자동차가 잘 움직이는지 확인한다.

### SLAM 실습(Cartographer)

주의: 

여기서부터는 만약 wsl 또는 가상머신을 이용해 실습하는 중이라면

느리거나 제대로 작동하지 않을 수 있습니다.

반드시 듀얼부팅 리눅스 환경에서 해주시기 바랍니다.

설치

1. Gazebo 설치

```bash
sudo apt install ros-humble-gazebo-*
```

2. Turtlebot3 패키지 설치

```bash
sudo apt install ros-humble-turtlebot3 ros-humble-turtlebot3-gazebo
```

3. Cartographer 설치

```bash
sudo apt install ros-humble-cartographer ros-humble-cartographer-ros
```

4. Navigation2 설치

```bash
sudo apt install ros-humble-navigation2
```

5. rtabmap 패키지 설치

```bash
sudo apt install ros-humble-rtabmap-ros
```

6. 압축 파일 다운(`realsense_warehouse_ws.zip`)
7. 워크스페이스 밖으로 나가서 압축 해제 (`realsense_warehouse_ws`라는 새로운 워크스페이스로 만들 예정)
8. `realsense_warehouse_ws` 워크스페이스로 이동 후 빌드
9. 환경변수 설정(로봇 모델 설정)

```bash
export TURTLEBOT3_MODEL=waffle
```

이는 터미널을 새로 열 때마다 매번 해줘야 하는 작업이며,

번거롭다면 아래 명령어로 bashrc 셸에 등록해주면 터미널을 열 때마다 자동으로 처리해준다.

```bash
echo 'export TURTLEBOT3_MODEL=waffle' >> ~/.bashrc
source ~/.bashrc
```

터미널을 3개를 띄워서

첫 번째 터미널에

```bash
ros2 launch turtlebot3_gazebo turtlebot3_no_roof_aws.launch.py
```

두 번째 터미널에

```bash
ros2 launch turtlebot3_cartographer cartographer.launch.py use_sim_time:=True
```

를 실행해본다.

마지막 터미널에는 (C)

```bash
ros2 run turtlebot3_teleop teleop_keyboard
```

를 실행하고 로봇을 조종해본다.

(조작법 : w 속도증가, x 속도감소, a, d 좌, 우회전 속도 증가)

(C) 터미널의 실행 코드를 종료하고

아래 명령어를 실행해 자동 탐색을 수행하게 한다.

```bash
ros2 run turtlebot3_gazebo turtlebot3_drive
```

혹여나 로봇이 길을 못찾고 헤메고 있다는 느낌이 들면

드라이브를 중단하고 키보드 코드로 수동 조작한 다음 다시 드라이브로 전환해도 된다.

rviz 상에서 어느정도 지도가 만들어지는지 확인하고

지도가 어느정도 완성되면 터미널을 하나 더 열어 아래 명령어로 지도를 저장한다.

```bash
source install/setup.bash
ros2 run nav2_map_server map_saver_cli -f ~/map
```

홈 디렉토리에 `map.pgm`과 `map.yaml`이 생겼는지 확인한다.

만약 폴더 안으로 경로를 정하고 싶다면

`~/map` 대신 `~/폴더이름/파일이름`으로 지정하면 된다.

.pgm 파일을 편집할 수 있게 해주는 gimp를 이용해 부정확한 부분을 수작업으로 보완해 줄 수도 있다.

설치 : `sudo apt install gimp`

#### 저장된 지도 실행하기

이제 실행중인 모든 코드를 종료하고

터미널 2개에 각각 아래 명령어를 실행한다.

```bash
ros2 launch turtlebot3_gazebo turtlebot3_no_roof_nav.launch.py
```

```bash
ros2 launch turtlebot3_navigation2 navigation2.launch.py use_sim_time:=True map:=$HOME/map.yaml
```

![Nav2](../imgs/gazebo/nav1.png)

2d pose estimate로 로봇의 현재 위치를 map에서 대략적으로 찍어주면 위 이미지처럼 map이 변한다.

이 상태에서 Nav2Goal을 눌러 목적지를 정해주면 로봇이 목적지로 가는 경로를 잡아주고 이동하게 된다.

Nav2Goal을 꾹 누른 채로 드래그를 하면 목적지에 도착했을 때의 로봇의 방향을 정해줄 수도 있다.